# nb17 — F15 Test: Continuous-Decay HDC-RWKV

### What this notebook tests

Per F13/F14 in [findings.md](../docs/findings.md):
- F13: pure-binary HDC-RWKV plateaus at BPC ~2.93 regardless of scale
- F14: relaxing the **output prototype** to int8 did not help (BPC stayed 2.94)

**F15 hypothesis**: the actual bottleneck is the **bipolar decay_mask** in the
recurrence. Bipolar `{-1, +1}` decay can only express two memory modes per
channel: 'keep' or 'flip every step'. Real RWKV uses continuous decay rates
per channel, letting different channels track different memory horizons.

### What changes here

| | Tier 3 (pure binary) | Tier 2.5 (int8 prototype) | **Tier 2.6 (this notebook)** |
|---|---|---|---|
| vocab_hv | bipolar | bipolar | bipolar (unchanged) |
| **decay** | **bipolar** | **bipolar** | **CONTINUOUS sigmoid ∈ (0,1)** |
| state | tanh | tanh | tanh (unchanged) |
| prototype | bipolar | int8 | int8 (unchanged from 2.5) |
| BPC | 2.93 | 2.94 | **?** |

### Decision rules

| Result | Interpretation | Paper finding |
|---|---|---|
| BPC < 2.5 | Decay was the bottleneck. Continuous decay breaks the ceiling. | F15: bipolar decay caused the limit; continuous decay reaches BPC X |
| BPC 2.5–2.7 | Significant relief but not full break | F15: partial — decay is one of multiple bottlenecks |
| BPC > 2.7 | Decay was NOT the bottleneck either | F15: architectural ceiling is wider than any single component |

### Storage at deployment

~149 KB (vocab 16 KB bipolar + decay 1 KB fp16 + prototype 131 KB int8).
Same as Tier 2.5 — decay is tiny vs the prototype matrix.


## Cell 1 — Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path

os.chdir('/kaggle/working')
REPO_URL = 'https://github.com/elixpo/wozformer.git'
subprocess.run(['rm', '-rf', '/kaggle/working/wozformer'], check=True)
subprocess.run(['git', 'clone', REPO_URL, '/kaggle/working/wozformer'], check=True)
WOZFORMER_PATH = Path('/kaggle/working/wozformer')
os.chdir(WOZFORMER_PATH)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.'], check=True)

# Force-evict cached modules
for m in list(sys.modules):
    if m.startswith('wozformer'):
        del sys.modules[m]

import wozformer as wz
import torch
import math
import matplotlib.pyplot as plt

print(f'wozformer {wz.__version__}, device: {wz.utils.get_device()}')
assert hasattr(wz.config, 'HDCRWKVContinuousDecayConfig'), 'config not found — kernel cache?'
assert hasattr(wz.models, 'HDCRWKVContinuousDecay'), 'model not found'


## Cell 2 — Hyperparameters

Identical to Tier 2.5 (nb16) for direct comparability. Only difference is
that the model class swaps bipolar decay for continuous decay.


In [ ]:
VOCAB_SIZE = 256
D          = 512
N_LAYERS   = 1
BLOCK_SIZE = 64

BATCH_SIZE = 32
LR         = 3e-3
N_STEPS    = 20000
EVAL_EVERY = 500
SEED       = 1337

RUN_DIR = Path('/kaggle/working/runs')
RUN_DIR.mkdir(parents=True, exist_ok=True)
OUT_PT  = RUN_DIR / 'tier2_6_cont_decay.pt'
BPE_JSON = RUN_DIR / 'bpe_256.json'


## Cell 3 — Corpus + BPE

In [ ]:
wz.utils.set_seed(SEED)
device = wz.utils.get_device()

text = wz.data.load_corpus(WOZFORMER_PATH / 'data' / 'tinyshakespeare.txt')
tok = wz.tokenizer.BPETokenizer.train(text, vocab_size=VOCAB_SIZE)
tok.save(BPE_JSON)

ids = torch.tensor(tok.encode(text), dtype=torch.long)
train_data, val_data = wz.data.split_train_val(ids)
print(f'tokens: train {len(train_data):,} / val {len(val_data):,}')


## Cell 4 — Build the continuous-decay variant

In [ ]:
cfg = wz.config.HDCRWKVContinuousDecayConfig(
    vocab_size=VOCAB_SIZE, d=D, n_layers=N_LAYERS, block_size=BLOCK_SIZE,
)
model = wz.models.HDCRWKVContinuousDecay(cfg).to(device)

n_train = wz.utils.count_params(model)
n_bytes = model.deployment_bytes()
print(f'trainable params: {n_train:,}')
print(f'deployment bytes: {n_bytes:,} ({n_bytes/1024:.1f} KB)')

# Verify decay init: should be sigmoid(2.0) ≈ 0.88 across all channels
decay0 = torch.sigmoid(model.decay_logits[0]).detach()
print(f'init decay: mean={decay0.mean().item():.3f}, range=[{decay0.min():.3f}, {decay0.max():.3f}]')
print('(should be ~0.88 — "mostly keep memory" initialisation)')

# Sanity
xb, yb = wz.data.make_batch(train_data, BATCH_SIZE, BLOCK_SIZE, device)
with torch.no_grad():
    _, loss = model(xb, yb)
print(f'init loss: {loss.item():.4f} (expect ~{math.log(VOCAB_SIZE):.4f})')


## Cell 5 — Train

~30–60 min on T4. The trainer tracks soft and hard val.

What we're hoping to see: val(hard) descends *below the Tier 2.5 floor of 4.31*
by step 10000. If the continuous decay was the bottleneck, BPC should drop
toward 2.0–2.4. If it plateaus near 4.31, decay wasn't the bottleneck either.


In [ ]:
train_cfg = wz.config.TrainConfig(
    batch_size=BATCH_SIZE, block_size=BLOCK_SIZE,
    lr=LR, n_steps=N_STEPS, eval_every=EVAL_EVERY,
    seed=SEED, weight_decay=0.0,
)
history, best = wz.trainer.train(
    model, train_data, val_data, train_cfg,
    device=device, eval_hard=True,
)
print(f'\nbest HARD val: {best["val"]:.4f} at step {best["step"]}')


## Cell 6 — Loss curves + BPC + decay distribution

Most informative plot: the histogram of learned decay rates. If they
converge to a wide distribution (some near 0, some near 1), the model
genuinely uses per-channel memory horizons. If they all collapse to one
value (or stay near init), the continuous flexibility wasn't useful.


In [ ]:
steps = [h[0] for h in history]
trains = [h[1] for h in history]
vals_soft = [h[2] for h in history]
vals_hard = [h[3] for h in history]

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].plot(steps, trains, label='train', alpha=0.6)
axes[0].plot(steps, vals_soft, label='val(soft)', alpha=0.8)
axes[0].plot(steps, vals_hard, label='val(hard, deployment)', linewidth=2, color='C2')
axes[0].axhline(4.35, color='red', linestyle='--', alpha=0.5, label='nb12c floor (4.35)')
axes[0].axhline(4.31, color='orange', linestyle='--', alpha=0.5, label='Tier 2.5 floor (4.31)')
axes[0].set_xlabel('step'); axes[0].set_ylabel('val (nats/token)')
axes[0].set_title('Continuous-decay variant vs prior ceilings')
axes[0].legend(); axes[0].grid(alpha=0.3)

final_decay = torch.sigmoid(model.decay_logits[0]).detach().cpu().numpy()
axes[1].hist(final_decay, bins=30, alpha=0.8)
axes[1].axvline(0.88, color='red', linestyle='--', alpha=0.5, label='init value (0.88)')
axes[1].set_xlabel('learned decay rate'); axes[1].set_ylabel('# channels')
axes[1].set_title(f'Per-channel decay distribution (d={D})')
axes[1].set_xlim(0, 1)
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

# BPC
sample = val_data[:5000].tolist()
avg_cpt = wz.metrics.avg_chars_per_token(tok, sample)
bpc = wz.metrics.bits_per_char(best['val'], avg_cpt)
print(f'avg chars/token: {avg_cpt:.2f}')
print(f'best HARD val:   {best["val"]:.4f} nats/token')
print(f'Tier 2.6 BPC:    {bpc:.4f}')
print()
print('Reference:')
print(f'  Teacher (fp32 transformer): BPC 2.04')
print(f'  Tier 2.5 (int8 proto):      BPC 2.94')
print(f'  nb12c (pure binary):        BPC 2.93')
if bpc < 2.5:
    print(f'\n  >> F15 POSITIVE. Continuous decay broke the ceiling: BPC {bpc:.2f}.')
    print(f'     Bipolar decay WAS the bottleneck of binary HDC-RWKV.')
elif bpc < 2.7:
    print(f'\n  >> F15 PARTIAL. BPC {bpc:.2f} — significant but not a full break.')
else:
    print(f'\n  >> F15 NEGATIVE. Decay wasn\'t the bottleneck either (BPC {bpc:.2f}).')
    print(f'     Architectural ceiling is wider than any single component.')

# Decay distribution analysis
d_std = final_decay.std()
d_min, d_max = final_decay.min(), final_decay.max()
print(f'\nDecay distribution: min={d_min:.3f}, max={d_max:.3f}, std={d_std:.3f}')
if d_std > 0.1:
    print('  Channels learned diverse decay rates — continuous flexibility used.')
else:
    print('  Decay collapsed to a narrow range — continuous flexibility unused.')


## Cell 7 — Generate samples

In [ ]:
prompts = ['king', 'romeo', 'my lord,', 'queen elizabeth:', 'to be or not']
seeds = [1337, 42, 7, 99, 2024]

for prompt, seed in zip(prompts, seeds):
    print(f'\n===== {prompt!r}  seed={seed} =====')
    out = wz.generate.generate(
        model, tok, prompt=prompt, max_new_tokens=120,
        block_size=BLOCK_SIZE, temperature=0.7, top_k=10,
        seed=seed, device=device, use_hard=True,
    )
    print(out)


## Cell 8 — Save checkpoint

Binary export deferred until we know whether this is the winning architecture.


In [ ]:
torch.save({
    'config': cfg.__dict__,
    'model_state': model.state_dict(),
    'history': history,
    'best_val_hard': best['val'],
    'best_step': best['step'],
    'deploy_bytes': n_bytes,
    'decay_distribution': torch.sigmoid(model.decay_logits[0]).detach().cpu().tolist(),
    'tier': 'tier2.6_continuous_decay',
}, OUT_PT)
print(f'saved → {OUT_PT}  ({OUT_PT.stat().st_size/1024:.1f} KB)')
print(f'\nDownload {OUT_PT.name} and {BPE_JSON.name} from /kaggle/working/runs/')


## Post-mortem

After running:
1. Paste the final BPC and the decay distribution histogram interpretation back
2. Paste the 5 generation samples
3. I'll log F15 with the verdict (positive / partial / negative)
4. Based on that, we'll either build the ESP32 binary export for this variant
   or proceed to firmware with the existing artifacts.
